# ML-02 - Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhalid04/Shaheer-Khalid-FlyRank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

My Week 1 writeup. Before I touch a model I want to pick a lane and frame the question properly. Plain words, and a few real numbers pulled straight from the starter data.


## 1. My lane (or freestyle) and why

I'm going with **Lane 2, Refresh / Content Opportunity Scoring**.

I picked it because it fits the data I actually have and a decision someone really makes. The starter file is one row per page with the last 90 days of search and engagement numbers, which is exactly what you need to ask which page is worth a person's time to look at next. The starter scripts (01 to 05) already run this lane start to finish, so I'm not building from zero. That lets me spend my seven weeks on the parts that are genuinely hard: writing an honest label that looks at the future instead of the current window, checking for leakage, and seeing whether a model can actually beat a plain rule.

I looked at the other lanes too. Lane 1 (signal analysis) mostly tells you which signals correlate and stops before an action list. Lane 4 (CTR) needs position adjustment before it can say anything I'd trust. Lane 3 (clustering) is more of a lens than a ranked decision. Lane 2 gives me a clear person to help (a content editor), a clear output (a ranked review queue with reasons attached), and room to make the label honest. The card says the lane stays provisional until Week 4, so I can still change my mind if the data pushes me somewhere else.


In [1]:
# Load the starter data once so the rest of the notebook can reuse it.
# The little loop just walks up to the repo root so this runs in Colab or locally.
import os
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"{df.shape[0]:,} pages x {df.shape[1]} columns")
print(f"{df['client_id'].nunique()} clients")
print("one row per page, which is the grain I want for 'review this one first?'")


30,000 pages x 44 columns
32 clients
one row per page, which is the grain I want for 'review this one first?'


## 2. The question: decision, action, cost of a wrong call

My question: using only signals we can see before anyone edits a page, which pages should a content editor look at first for a refresh?

A few things I want to be clear about up front:

- One row is one page (`content_id`), summed over the last 90 days. That's my unit.
- The output is a ranked list. Each page gets a score plus a couple of short reason tags a person can actually read.
- The decision it helps is where a small content team spends the few review hours it has, meaning which page to open first.
- The person acting is a content editor. They work down the list and refresh, expand, prune, or just keep watching each page.
- If I'm wrong the cost goes two ways. A false positive burns an editor's time on a page that was fine, and it might even disturb a page that didn't need touching. A false negative leaves a page that's really slipping, but still has demand, to keep losing traffic while nobody looks. Since the team can only get through a handful of pages, the expensive mistake is putting the wrong pages at the top of the list. So the metric that matches this is precision at K, not plain accuracy.
- Do I even need a model? A simple rule like "old and still getting impressions" already catches some of these, and I'll build that as my baseline. But the things that mark a good refresh candidate (demand, movement, position, freshness, engagement) pull in different directions on different pages. That's the one case where a learned ranking can earn its keep. I'll only say it does if it beats my rule under honest validation.


In [2]:
# Show that the candidate pool is already way bigger than any review team's capacity.
declining = df["trend_direction"] == "down"
declining_with_demand = declining & (df["impressions_90d"] >= 100)

n = int(declining_with_demand.sum())
print(f"Pages trending down and still in demand (>=100 impressions/90d): {n:,} "
      f"({declining_with_demand.mean()*100:.1f}% of all pages)")
print(f"At ~50 reviews a week, an editor would need ~{n/50:,.0f} weeks just to clear that pool by hand.")
print("so the order of the list is the whole game, and precision at K is what matches it.")


Pages trending down and still in demand (>=100 impressions/90d): 13,152 (43.8% of all pages)
At ~50 reviews a week, an editor would need ~263 weeks just to clear that pool by hand.
so the order of the list is the whole game, and precision at K is what matches it.


## 3. Quick look at the data (2-3 real numbers)

Here are three numbers from the starter file (the cell below works them out live) that made me think this lane is worth seven weeks:

1. There are 30,000 pages across 32 clients. Nobody reviews that by hand, so a ranked queue isn't a nice-to-have here, it's the only way the work gets done.
2. About 13,152 pages, roughly 43.8%, are trending down and still have real demand (at least 100 impressions in 90 days). That's the pool where a refresh might actually pay off, and it's far too big to eyeball. That's the "which one first?" problem in a nutshell.
3. Traffic is lopsided. The top 10% of pages hold about 70% of all impressions. So effort should follow the traffic, which is a ranking job, and getting the order right genuinely changes the outcome.

One caution I want to flag now: `trend_direction == "down"` is just a bucket from the current window, not a real future outcome. I'm only using it here to show the candidate pool is large. Section 4 says why my capstone needs a stronger, future-looking label.


In [3]:
# The three numbers I'm leaning on for the lane choice.
n_pages   = len(df)
n_clients = df["client_id"].nunique()

declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)

imp = df["impressions_90d"].sort_values(ascending=False)
top10_share = imp.head(int(len(df) * 0.10)).sum() / imp.sum()

print(f"1. Scale:          {n_pages:,} pages across {n_clients} clients")
print(f"2. Candidate pool: {int(declining_with_demand.sum()):,} pages "
      f"({declining_with_demand.mean()*100:.1f}%) declining with demand (>=100 impressions/90d)")
print(f"3. Concentration:  top 10% of pages hold {top10_share*100:.1f}% of all impressions")


1. Scale:          30,000 pages across 32 clients
2. Candidate pool: 13,152 pages (43.8%) declining with demand (>=100 impressions/90d)
3. Concentration:  top 10% of pages hold 70.2% of all impressions


## 4. Careful words: what I can and can't claim

What I think I can honestly say:

- Observed and directional only. Something like "pages with these traits tend to be the ones that later show up as declining", never "this trait causes decline".
- Decision support. My list says which pages a person should look at first when they can't get to all of them. It does not promise a page will bounce back if someone edits it.
- Beats a baseline, if it actually does. If the model wins, I'll say it beat a plain rule on this data under a client-holdout split (and later a time-aware one), not that it's some universal SEO tool.

What I won't claim:

- That I proved a Google ranking factor. This is data I watched, not an experiment I ran.
- That a refresh caused a recovery. That needs an experiment or a proper causal setup, and I don't have one here.
- Anything about AI citations or AI rankings. The data only counts click-through sessions from AI tools, and those are rare.
- That the starter label is the truth. `trend_direction` comes straight from `trend_pct`, so both are off limits as features (that would be leakage). My capstone will define a future-window outcome instead, with features taken from an earlier window and a strict leakage check.
- I'll keep everything public safe: no client names, domains, URLs, or raw queries, just pseudonymized IDs and aggregates.


In [4]:
# Quick check on why trend_direction and trend_pct can't be features.
# trend_direction is just a bucket of trend_pct, and trend_pct is the label source.
proxy = df.dropna(subset=["trend_pct"]).groupby("trend_direction")["trend_pct"].agg(["min", "max", "count"])
print("trend_pct range inside each trend_direction bucket:")
print(proxy.round(1).to_string())
print()
print("'down' is basically trend_pct <= -20%. The label is the feature, so using either would leak the answer.")
print("Fix for the capstone: define a future-window outcome and keep the feature window strictly before it.")


trend_pct range inside each trend_direction bucket:
                   min      max  count
trend_direction                       
down            -100.0    -20.0  16262
stable           -20.0     20.0   5962
up                20.0  44900.0   4388

'down' is basically trend_pct <= -20%. The label is the feature, so using either would leak the answer.
Fix for the capstone: define a future-window outcome and keep the feature window strictly before it.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.
